In [8]:
import pandas as pd
import numpy as np
from pathlib import Path

In [11]:
pd.set_option('display.float_format', '{:.4f}'.format)

In [26]:
####################
###### Reading Data ##############    
####################    

AllDataset = pd.read_csv('../dataset/dataset_with_outlier_removed.csv', low_memory=False)
AllDataset = AllDataset.set_index(AllDataset.ID)


In [27]:
column_list = ["DX", "DX6", "DX12", "DX24"]

# Dictionary to store value counts for each column
value_counts_dict = {}

for col in column_list:
    value_counts_dict[col] = AllDataset[col].value_counts()

# Combine into a single DataFrame
result_df = pd.DataFrame(value_counts_dict).fillna(0).astype(int)

# Show the result
print(result_df)

       DX   DX6  DX12  DX24
AD   2187  2363  2495  2703
MCI  2033  1856  1712  1520
CN   1373  1374  1386  1370


In [28]:
# Example column list
column_list = ["DX", "DX6", "DX12", "DX24"]

# Function to label if diagnosis changed
def label_dx_change(row):
    initial = row["DX"]
    # Check if any future timepoints are different from the initial
    changed = any(pd.notna(row[col]) and row[col] != initial for col in column_list[1:])
    return f"{initial}_c" if changed else initial

# Apply function to create a new column
AllDataset["DX_status"] = AllDataset.apply(label_dx_change, axis=1)

# Count each unique label
dx_status_counts = AllDataset["DX_status"].value_counts()

print(dx_status_counts)


DX_status
AD       2176
MCI      1434
CN       1295
MCI_c     599
CN_c       78
AD_c       11
Name: count, dtype: int64


In [29]:
rid_counts = AllDataset.groupby('RID').size()

# Calculate statistics
min_count = rid_counts.min()
median_count = rid_counts.median()
mean_count = rid_counts.mean()
max_count = rid_counts.max()

# Display results
print(f"Min count: {min_count}")
print(f"Median count: {median_count}")
print(f"Mean count: {mean_count}")
print(f"Max count: {max_count}")

Min count: 1
Median count: 3.0
Mean count: 3.5806658130601794
Max count: 13


In [30]:
column_list = ["DX", "DX6", "DX12", "DX24"]
time_transitions = list(zip(column_list[:-1], column_list[1:]))

# Dictionary to hold conversion counts per timepoint
timepoint_conversion_counts = {}

for prev_col, curr_col in time_transitions:
    transition_series = AllDataset[[prev_col, curr_col]].dropna()
    
    # Filter rows where diagnosis changes
    transition_series = transition_series[transition_series[prev_col] != transition_series[curr_col]]
    
    # Create conversion label
    transitions = transition_series[prev_col] + "_" + transition_series[curr_col]
    
    # Count transitions
    transition_counts = transitions.value_counts()
    
    # Store in dictionary
    timepoint_conversion_counts[f"{prev_col}->{curr_col}"] = transition_counts

# Display results
for timepoint, counts in timepoint_conversion_counts.items():
    print(f"\nConversions for {timepoint}:")
    print(counts)


Conversions for DX->DX6:
MCI_AD    182
MCI_CN     12
CN_MCI     11
AD_MCI      6
Name: count, dtype: int64

Conversions for DX6->DX12:
MCI_AD    136
MCI_CN     21
CN_MCI      9
AD_MCI      4
Name: count, dtype: int64

Conversions for DX12->DX24:
MCI_AD    214
CN_MCI     64
MCI_CN     51
AD_MCI      9
CN_AD       3
Name: count, dtype: int64


In [31]:
import pandas as pd

regions = ['Ventricles/ICV', 'Hippocampus/ICV', 'WholeBrain/ICV', 'Entorhinal/ICV', 'Fusiform/ICV', 'MidTemp/ICV',
           'FDG', 'AV45',
           'CDRSB', 'MMSE', 'RAVLT_immediate', 'RAVLT_learning', 'RAVLT_forgetting','RAVLT_perc_forgetting','FAQ', 'MOCA', 'LDELTOTAL', 'DIGITSCOR','TRABSCOR',
           'ABETA', 'PTAU', 'TAU', 'AGE', 'PTEDUCAT', 'APOE4']
dx_order = ['CN', 'MCI', 'AD']  # Desired DX order

# Collect stats
summary_rows = []

for dx, group in AllDataset.groupby('DX'):
    for region in regions:
        values = group[region]
        summary_rows.append({
            'Region': region,
            'DX': dx,
            'median': values.median(),
            "min": values.min(),
            'max': values.max(),
            'missing': values.isna().sum()
        })

# Create DataFrame
summary_df = pd.DataFrame(summary_rows)

# Convert DX to categorical with order
summary_df['DX'] = pd.Categorical(summary_df['DX'], categories=dx_order, ordered=True)

# Sort by Region, then DX
summary_df = summary_df.sort_values(by=['Region', 'DX'])

# Set MultiIndex and round
summary_df.set_index(['Region', 'DX'], inplace=True)
summary_df = summary_df.round(4)

output_path = Path('../results/eda/dataset_summary_table_for_paper.csv')
output_path.parent.mkdir(parents=True, exist_ok=True)
summary_df.to_csv(output_path)
# Show the final result
print(summary_df)


                      median      min       max  missing
Region         DX                                       
ABETA          CN  1267.0000 200.0000 1700.0000      783
               MCI  833.1000 213.1000 1700.0000     1230
               AD   570.8500 212.3000 1700.0000     1669
AGE            CN    76.3000  56.2000   93.6000        0
               MCI   74.8000  55.0000   94.9000        0
...                      ...      ...       ...      ...
Ventricles/ICV MCI    0.0238   0.0045    0.0832      371
               AD     0.0318   0.0072    0.0880      680
WholeBrain/ICV CN     0.6732   0.5299    0.8260      248
               MCI    0.6637   0.4851    0.8537      320
               AD     0.6199   0.4742    0.7760      617

[75 rows x 4 columns]
